#1 setting

In [1]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
import matplotlib.pyplot as plt
import imageio
import os
from IPython.display import Image

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


#2 define NN(Dueling DQN)

In [2]:
class DuelingDQN(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=128):
        super(DuelingDQN, self).__init__()
        self.feature = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU()
        )
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, action_size)
        )

    def forward(self, state):
        features = self.feature(state)
        values = self.value_stream(features)
        advantages = self.advantage_stream(features)
        # Q = V + (A - mean(A))
        qvals = values + (advantages - advantages.mean(dim=1, keepdim=True))
        return qvals

#3buffer

In [3]:
class DQNAgent:
    def __init__(self, state_size, action_size, lr=1e-3, gamma=0.99,
                 tau=0.005, buffer_size=100000, batch_size=64,
                 epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = gamma
        self.tau = tau
        self.batch_size = batch_size
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay

        # 网络
        self.q_network = DuelingDQN(state_size, action_size).to(device)
        self.target_network = DuelingDQN(state_size, action_size).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)

        # 经验回放
        self.memory = ReplayBuffer(buffer_size)

        # 记录损失
        self.losses = []

    def act(self, state, eval_mode=False):
        if not eval_mode and np.random.rand() < self.epsilon:
            return np.random.randint(self.action_size)
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = self.q_network(state)
        return q_values.argmax().item()

    def step(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)
        if len(self.memory) >= self.batch_size:
            self.learn()

    def learn(self):
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        # Double DQN: 用在线网络选择动作，用目标网络评估 Q 值
        with torch.no_grad():
            next_actions = self.q_network(next_states).argmax(dim=1, keepdim=True)
            next_q = self.target_network(next_states).gather(1, next_actions)
            target_q = rewards + (1 - dones) * self.gamma * next_q

        current_q = self.q_network(states).gather(1, actions)
        loss = nn.MSELoss()(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # 记录损失
        self.losses.append(loss.item())

        # 软更新目标网络
        for target_param, param in zip(self.target_network.parameters(), self.q_network.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

        # 衰减 epsilon
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

#4 training

In [4]:
def train(env_name="LunarLander-v3", episodes=1000):
    env = gym.make(env_name, render_mode="rgb_array")  # 用于录制
    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n

    agent = DQNAgent(state_size, action_size)

    episode_rewards = []
    episode_lengths = []

    # 用于绘制损失
    all_losses = []

    for ep in range(episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False
        step = 0

        while not done:
            action = agent.act(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            agent.step(state, action, reward, next_state, done)
            state = next_state
            total_reward += reward
            step += 1

        episode_rewards.append(total_reward)
        episode_lengths.append(step)
        all_losses.extend(agent.losses)
        agent.losses = []  # 每个 episode 后清空，便于按 episode 统计平均损失

        if (ep+1) % 100 == 0:
            avg_reward = np.mean(episode_rewards[-100:])
            print(f"Episode {ep+1}/{episodes}, Avg Reward (last 100): {avg_reward:.2f}, Epsilon: {agent.epsilon:.3f}")

    env.close()
    return agent, episode_rewards, episode_lengths, all_losses

#5 run

In [5]:
agent, rewards, lengths, losses = train(episodes=1000)

# 绘制奖励曲线
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(rewards, alpha=0.3, label='Episode Reward')
# 滑动平均
window = 50
smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(rewards)), smoothed, 'r', label='Smoothed (50)')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Training Rewards')
plt.legend()

# 绘制损失曲线（每步损失，可下采样）
plt.subplot(1,2,2)
plt.plot(losses, alpha=0.5, label='Loss per update')
# 平滑
smoothed_loss = np.convolve(losses, np.ones(100)/100, mode='valid')
plt.plot(range(99, len(losses)), smoothed_loss, 'r', label='Smoothed (100)')
plt.xlabel('Training Step (update)')
plt.ylabel('MSE Loss')
plt.title('Training Loss')
plt.legend()
plt.tight_layout()
plt.savefig('training_curves.png')
plt.show()

NameError: name 'ReplayBuffer' is not defined

#6 result

In [ ]:
def record_episode(agent, env_name="LunarLander-v3", gif_path="lander.gif"):
    env = gym.make(env_name, render_mode="rgb_array")
    frames = []
    state, _ = env.reset()
    done = False
    while not done:
        frames.append(env.render())
        action = agent.act(state, eval_mode=True)  # 评估模式（不探索）
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
    env.close()
    # 保存为 GIF
    imageio.mimsave(gif_path, frames, fps=30)
    print(f"GIF saved to {gif_path}")

record_episode(agent)
# 在 notebook 中显示
Image(open('lander.gif','rb').read())